In [4]:
import requests
import numpy as np
import pandas as pd
import sqlite3
from datetime import datetime, timezone

In [5]:
conn = sqlite3.connect("screener.db", timeout=20)
cursor = conn.cursor()
conn.execute("PRAGMA journal_mode=WAL;")

cursor.execute("""
    CREATE TABLE IF NOT EXISTS pool_snapshots (
        id               INTEGER PRIMARY KEY AUTOINCREMENT,
        pair_address     TEXT REFERENCES pools(pair_address),
        price_usd        REAL,
        liquidity_usd    REAL,
        volume_m5        REAL,
        volume_h1        REAL,
        volume_h24       REAL,
        price_change_m5  REAL,
        price_change_h1  REAL,
        price_change_h24 REAL,
        market_cap       REAL,
        fdv              REAL,
        snapshot_at      TIMESTAMP
    )
""")
conn.commit()

In [6]:
cursor.execute("""
    SELECT pair_address, token_address, chain_id
    FROM pools
""")
pools = cursor.fetchall()
snapshot_count = 0

for pair_address, token_address, chain_id in pools:

    response = requests.get(
        f"https://api.dexscreener.com/token-pairs/v1/{chain_id}/{token_address}"
    )

    pair_response = response.json()
    pool = next((p for p in pair_response if p.get("pairAddress") == pair_address), None)

    if pool is None:
        continue

    now = datetime.now(timezone.utc).isoformat()

    cursor.execute(
        """
        INSERT INTO pool_snapshots(
            pair_address,
            price_usd,
            liquidity_usd,
            volume_m5,
            volume_h1,
            volume_h24,
            price_change_m5,
            price_change_h1,
            price_change_h24,
            market_cap,
            fdv,
            snapshot_at
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """,
        (
            pair_address,
            pool.get("priceUsd"),
            pool.get("liquidity", {}).get("usd"),
            pool.get("volume", {}).get("m5"),
            pool.get("volume", {}).get("h1"),
            pool.get("volume", {}).get("h24"),
            pool.get("priceChange", {}).get("m5"),
            pool.get("priceChange", {}).get("h1"),
            pool.get("priceChange", {}).get("h24"),
            pool.get("marketCap"),
            pool.get("fdv"),
            now,
        ),
    )
    snapshot_count += 1

conn.commit()
print(f"Snapshots inserted: {snapshot_count}")
conn.close()

Snapshots inserted: 84
